In [ ]:
import numpy as np
import pickle
import time 
import timeit
import shutil 
import h5py
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm 
from functools import partial
import glob
import os
import pandas as pd
import statistics as sta
from scipy.stats import circstd, circmean
from scipy.signal import savgol_filter
from matplotlib.animation import FuncAnimation
#import changefinder
import ruptures as rpt
import seaborn as sns

from matplotlib import animation

from pathlib import Path


In [ ]:
loc = "~/vmodel_output/"
file = "noPred_noMultCol_v2pol_astr_dphi.csv"
method="bs"
plt.style.use('default')
pol = pd.read_csv(loc+file, header=None, index_col=False)
pol2 = pd.read_csv(loc+file, header=None, index_col=False)
relax_T3=[]

In [ ]:
pol

In [ ]:
## ALL THE CODE BELOW - COMPUTES THE RELAXATION TIME ###
mean_pol0=np.transpose(pol2).mean(axis=1)
print(mean_pol0)
mean_pol1=[np.mean(mean_pol0)]*time
flat_list = [item for sublist in pol2 for item in sublist]
std_pol1=np.std(flat_list)
    
for k in range(nruns):
           # t_end_fnt=[index for index,value in enumerate(FNT[k]) if value == 1][-1]
    points=pol[k]         
    if method=="dp":
                #Changepoint detection with dynamic programming search method
        model = "l1"  
        algo = rpt.Dynp(model=model, min_size=3, jump=5).fit(points)
    elif method=="bs":
                #Changepoint detection with the Binary Segmentation search method
        model = "l2"  
        algo = rpt.Binseg(model=model).fit(points)
    my_bkps = algo.predict(n_bkps=10)
          #  relax_t[k] = my_bkps[-2]-t_end_fnt  #### <----- WE DO NOT DO IT SINCE IT CAN INCREASE THE VARIATION IN THE VALUES, WE WANT TO KEEP IT CLEAN
           
            #############################################################################
            #############################################################################
           #### Lower Control Limits
    yhat2 = savgol_filter(mean_pol1-3*std_pol1-points, 333, 2) #smoothes 
    l2=np.where(yhat2 <0)[0]
    if len(np.array(l2))>0:
        i=1
        while (l2[i]-l2[i-1]==1) and (i<len(np.array(l2))-1):
                      i=i+1
    else:
        i=0

    if i==len(np.array(l2))-1 or i==0:
        relax3_t[k]=np.nan # time
    else:
        relax3_t[k]=l2[i]


relax_T3.append(relax3_t)

In [ ]:
method="bs"
plt.style.use('default')

#folders = ['/FrontA-40/', '/BackA-40/', '/SideA-40/']
folders = ['/FRONT/', '/BACK/', '/SIDE/']
cwd = os.getcwd()
#### UPLOADING DATA WITHOUT THE PREDATPR #################
#nopred_files=glob.glob('/extra2/pbartash/NoPred/*.h5')
nopred_files=glob.glob('/home/bartashevich/Documents/NoPred/*.h5')

relax_T=[]
relax_T3=[] #### YOU WILL NEED ONLY THIS #################
NND=[]
allMD=[]

In [ ]:
for atk_id in range(0,len(folders)):

    #files = glob.glob(cwd+ folders[atk_id] + '/*Dphi0.2.h5')
    files = glob.glob(cwd+ '/Exp5' + folders[atk_id] + '*Dphi0.2.h5')
    
    sorted_files=np.sort(files)
    sfiles=sorted_files #[dphi::11]

    psorted_files=np.sort(nopred_files)
    psfiles=psorted_files[2]
    m2 = h5py.File(psfiles, "r") # FILE WITHOUT A PREDATOR - ONLY ONE, SINCE THERE ARE NO ATTACKS FROM ANY DIRECTIONS

    for id in range(len(sfiles)):
        m = h5py.File(sfiles[id], "r") # FILE WITH THE PREDATOR, ITERATED OVER FRONT, BACK AND SIDE ATTACKS IN THE FOR-LOOP WITH "ID"
       
        for key in m.keys():
            print(key)

        for key2 in m2.keys():
            print(key2)

        dyn_runs=m[key]["swarm"] # WITH A PREDATOR
        dyn_runs2=m2[key2]["swarm"] # WITHOUT A PREDATOR

        nruns=dyn_runs.shape[0]
        tt=  dyn_runs2.shape[1] #dyn_runs.shape[1]

        print(dyn_runs.shape[0])
        print(dyn_runs.shape[1])
        print(dyn_runs2.shape[0])
        print(dyn_runs2.shape[1])

       ## analysis of foun data: time of fountain effect
    
        time = tt

        MD=[[]]*nruns
        FNT=[[]]*nruns
        fnt_start=[[]]*nruns
        fnt_end=[[]]*nruns
        nnd=[[]]*nruns
        iid=[[]]*nruns
        pol=[[]]*nruns
        pol2=[[]]*nruns
        elong=[[]]*nruns
        relax_t = [[]]*nruns
        relax2_t = [[]]*nruns
        relax3_t = [[]]*nruns

        for fn in range(nruns):
            for i in range(time):
                nnd[fn]=np.append(nnd[fn],dyn_runs[fn][i][13])
                iid[fn]=np.append(iid[fn],dyn_runs[fn][i][14])
                pol[fn]=np.append(pol[fn],dyn_runs[fn][i][1]) # <----- WHAT YOU NEED: POLARISATION WITH THE PREDATOR
              # pol2[fn]=np.append(pol2[fn],dyn_runs2[fn][i][1])
                elong[fn]=np.append(elong[fn],dyn_runs[fn][i][9])
        
        
        for fn in range(nruns):
            for i in range(time):
                pol2[fn]=np.append(pol2[fn],dyn_runs2[fn][i][1]) # <----- WHAT YOU NEED: POLARISATION WITHOUT THE PREDATOR

        
        ################################################################################
        ################## THE IMPORTANT STUFF STARTS BELOW ############################
        ################################################################################

        ## ALL THE CODE BELOW - COMPUTES THE RELAXATION TIME ###
        mean_pol0=np.transpose(pol2).mean(axis=1)
        mean_pol1=[np.mean(mean_pol0)]*time
        flat_list = [item for sublist in pol2 for item in sublist]
        std_pol1=np.std(flat_list)
    
        for k in range(nruns):
           # t_end_fnt=[index for index,value in enumerate(FNT[k]) if value == 1][-1]
            points=pol[k]         
            if method=="dp":
                #Changepoint detection with dynamic programming search method
                model = "l1"  
                algo = rpt.Dynp(model=model, min_size=3, jump=5).fit(points)
            elif method=="bs":
                #Changepoint detection with the Binary Segmentation search method
                model = "l2"  
                algo = rpt.Binseg(model=model).fit(points)
            my_bkps = algo.predict(n_bkps=10)
          #  relax_t[k] = my_bkps[-2]-t_end_fnt  #### <----- WE DO NOT DO IT SINCE IT CAN INCREASE THE VARIATION IN THE VALUES, WE WANT TO KEEP IT CLEAN
           
            #############################################################################
            #############################################################################
           #### Lower Control Limits
            yhat2 = savgol_filter(mean_pol1-3*std_pol1-points, 333, 2) #smoothes 
            l2=np.where(yhat2 <0)[0]
            if len(np.array(l2))>0:
                i=1
                while (l2[i]-l2[i-1]==1) and (i<len(np.array(l2))-1):
                      i=i+1
            else:
                i=0

            if i==len(np.array(l2))-1 or i==0:
                relax3_t[k]=np.nan # time
            else:
                relax3_t[k]=l2[i]


        relax_T3.append(relax3_t)

